# 第 5 周练习：专家知识工作者 (RAG)

该笔记本为保险科技公司 Insurellm 实现了 **RAG（检索增强生成）** 管道。它结合了所有 5 天的学习内容：

- **第 1 天**：具有基于关键字的上下文检索的简单 RAG
- **第 2 天**：文档分块、矢量嵌入（色度）和可视化
- **第 3 天**：与 LangChain 合作的完整 RAG 管道（检索器 + 法学硕士）
- **第 4 天**：评估（检索指标 + 法学硕士作为法官）
- **第 5 天**：高级技术（可选：重新排名、查询重写）

### 要求
- 从 **week5** 目录运行：`cd week5 && jupyter notebook`
- 或从存储库根运行 - 设置单元将自动配置路径
- 确保“.env”具有“OPENAI_API_KEY”（以及用于 HuggingFace 嵌入的可选“HF_TOKEN”）

＃＃ 设置

In [ ]:
# 确保我们在 week5 目录中，以便解决实施、评估和知识库问题
import sys
import os
from pathlib import Path

repo_root = Path.cwd()
if (repo_root / "week5").exists():
    week5_dir = repo_root / "week5"
else:
    week5_dir = repo_root  # Already in week5
sys.path.insert(0, str(week5_dir))
os.chdir(week5_dir)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# 下方为可执行代码（逻辑与字符串保持原文，便于运行）
import glob
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.documents import Document
import gradio as gr

load_dotenv(override=True)

MODEL = "gpt-4.1-nano"
DB_NAME = "vector_db"

openai_api_key = os.getenv("OPENAI_API_KEY")
if openai_api_key:
    print(f"OpenAI API Key found (starts with {openai_api_key[:8]}...)")
else:
    print("OPENAI_API_KEY not set — add it to .env")

## A 部分：文档摄取和分块

从知识库加载文档并将其分割成块以进行嵌入。

In [ ]:
# 从知识库加载文档（员工、产品、合同、公司）
folders = glob.glob("knowledge-base/*")
documents = []
for folder in folders:
    doc_type = os.path.basename(folder)
    loader = DirectoryLoader(
        folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={"encoding": "utf-8"}
    )
    folder_docs = loader.load()
    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)

print(f"Loaded {len(documents)} documents")

In [ ]:
# 使用 RecursiveCharacterTextSplitter 对文档进行分块（第 2 天风格）
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)
print(f"Created {len(chunks)} chunks")
print(f"Sample chunk:\n{chunks[0].page_content[:300]}...")

## B 部分：矢量存储（Chroma）

将块编码为向量并存储在 Chroma 中。使用 HuggingFace `all-MiniLM-L6-v2`（免费）或 OpenAI 嵌入。

In [ ]:
# 使用 HuggingFace 嵌入（免费） - 或 OpenAI 以获得更高的质量
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
# 嵌入 = OpenAIEmbeddings(model="text-embedding-3-large")

# 重建矢量存储（如果存在，则删除现有矢量存储）
if os.path.exists(DB_NAME):
    Chroma(persist_directory=DB_NAME, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(
    documents=chunks, embedding=embeddings, persist_directory=DB_NAME
)
print(f"Vector store created with {vectorstore._collection.count()} documents")

### 可选：可视化向量（第 2 天风格）

使用 t-SNE 将嵌入减少到 2D/3D，并按文档类型进行绘图。

In [ ]:
# 下方为可执行代码（逻辑与字符串保持原文，便于运行）
import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go

collection = vectorstore._collection
result = collection.get(include=["embeddings", "documents", "metadatas"])
vectors = np.array(result["embeddings"])
doc_types = [m.get("doc_type", "unknown") for m in result["metadatas"]]
colors = {"products": "blue", "employees": "green", "contracts": "red", "company": "orange"}
color_list = [colors.get(t, "gray") for t in doc_types]

tsne = TSNE(n_components=2, random_state=42)
reduced = tsne.fit_transform(vectors)
fig = go.Figure(data=[go.Scatter(
    x=reduced[:, 0], y=reduced[:, 1], mode="markers",
    marker=dict(size=5, color=color_list, opacity=0.8),
    text=doc_types, hoverinfo="text"
)])
fig.update_layout(title="2D Vector Store Visualization", width=700, height=500)
fig.show()

## C 部分：RAG 管道（第 3 天）

连接检索器和 LLM 以使用检索到的上下文回答问题。

In [ ]:
# 下方为可执行代码（逻辑与字符串保持原文，便于运行）
retriever = vectorstore.as_retriever(k=10)
llm = ChatOpenAI(temperature=0, model_name=MODEL)

SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing Insurellm.
Use the given context to answer questions. If you don't know the answer, say so.
Context:
{context}
"""

def answer_question(question: str, history=None):
    history = history or []
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    messages = [SystemMessage(content=system_prompt)]
    for h in history:
        if h.get("role") == "user":
            messages.append(HumanMessage(content=h["content"]))
    messages.append(HumanMessage(content=question))
    response = llm.invoke(messages)
    return response.content, docs

In [ ]:
# 测试 RAG 管道
answer, docs = answer_question("Who founded Insurellm?")
print("Q: Who founded Insurellm?")
print(f"A: {answer}")
print(f"\nRetrieved {len(docs)} chunks")

## D 部分：评估（第 4 天）

对测试问题样本进行检索和答案评估。完整的评估模块（“evaluation.eval”）使用“implementation.answer”——为此，请确保您已从 week5 目录运行“implementation/ingest.py”。这里我们直接使用笔记本的 RAG 进行评估。

In [ ]:
# 评估评估套件中的一些测试问题
from evaluation.test import load_tests

tests = load_tests()
sample = tests[:3]  # First 3 tests

for t in sample:
    answer, docs = answer_question(t.question)
    print(f"Q: {t.question}")
    print(f"A: {answer}")
    print(f"Reference: {t.reference_answer}")
    print("-" * 60)

## E 部分：Gradio 聊天界面

Insurellm 专家助理的交互式问答界面。

In [ ]:
def chat_fn(message, history):
    history = history or []
    # 将 Gradio 历史记录转换为字典列表（处理消息格式和元组格式）
    prior = []
    for h in history:
        if isinstance(h, dict):
            prior.append({"role": h.get("role", "user"), "content": h.get("content", str(h))})
        elif isinstance(h, (list, tuple)) and len(h) >= 2:
            prior.append({"role": "user", "content": h[0]})
            prior.append({"role": "assistant", "content": h[1]})
    answer, _ = answer_question(message, prior)
    return answer

gr.ChatInterface(chat_fn, title="Insurellm Expert Assistant", type="messages").launch(inbrowser=True)